# UniChart — Formatting Settings Tests

Confirms the dataset- and variable-level formatting setters behave as intended:

- `color`, `marker`, `linestyle`, `markersize`, `linewidth`, `alpha`
- **Autodetection**: an int / `'all'` / list-of-ints / `Dataset` selects
  *dataset(s)*; a **string** (or list of strings) targets a *variable/parameter*
  via `var_format`.
- **Negative indexing**: `-1` is the last dataset.
- **Precedence**: a variable override beats the dataset attribute at plot time.
- `'reset'` / `clear_var_format` / `reset_format`.
- Int-normalization (palette / marker map) on both the dataset and variable paths.
- Dataset **titles are no longer selectors**.

Each section makes hard `assert`s (the real test) and renders a plot for visual
confirmation. Note: dataset-level styling shows in `plot()`, while
variable-level styling shows in `plot_ymult()` (and `bar()`) — the test uses the
appropriate one in each section.

A summary cell at the end prints **ALL FORMATTING TESTS PASSED** if every
assertion held.

In [1]:
# --- make repo-root importable (notebook lives in demo_notebooks/) ---
import sys, os
_repo_root = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

import numpy as np
import pandas as pd
import plotly.express as px
import unichart as U
from unichart import UnichartNotebook, marker_map, _resolve_var_format

PASSED = 0
def check(cond, msg):
    """Assert + log a PASS line."""
    global PASSED
    assert cond, 'FAILED: ' + msg
    PASSED += 1
    print('PASS:', msg)

def make_nb():
    """Three datasets (A, B, C), each with Temp / Press / Flow variables."""
    np.random.seed(1)
    x = np.linspace(0, 10, 30)
    nb = UnichartNotebook()
    for i, name in enumerate('ABC'):
        df = pd.DataFrame({
            'x': x,
            'Temp':  np.sin(x) + i,
            'Press': np.cos(x) + i,
            'Flow':  0.5 * x + i,
        })
        nb.load_df(df, title=name)
    return nb

# Capture the factory-default dataset attributes for 'untouched' assertions.
# (color is assigned per-dataset from the palette, so it is handled separately.)
_d = make_nb().sets[0]
DEFAULTS = {k: getattr(_d, k) for k in
            ('marker', 'linestyle', 'markersize', 'linewidth', 'alpha')}
print('default dataset attrs (uniform across sets):', DEFAULTS)

UniChart Notebook Environment Initialized.
Loaded Set 0: A
Loaded Set 1: B
Loaded Set 2: C
default dataset attrs (uniform across sets): {'marker': 'o', 'linestyle': None, 'markersize': 10, 'linewidth': 2, 'alpha': 1}


## A. Dataset-level formatting (int / `'all'`)

Integer and `'all'` selectors write the dataset attributes and must **not**
touch `variable_formats`.

In [2]:
uc = make_nb()

uc.marker('all', 'o')
uc.linestyle('all', '-')
uc.color(0, 'red')
uc.color(1, 'green')
uc.color(2, 'blue')
uc.markersize(0, 14)
uc.linewidth(1, 4)
uc.alpha(2, 0.4)

check(uc.sets[0].color == 'red', "color(0,'red') sets dataset 0")
check(uc.sets[1].color == 'green', "color(1,'green') sets dataset 1")
check(uc.sets[2].color == 'blue', "color(2,'blue') sets dataset 2")
check(all(d.marker == 'o' for d in uc.sets), "marker('all','o') sets every dataset")
check(uc.sets[0].markersize == 14, "markersize(0,14) sets dataset 0")
check(uc.sets[1].linewidth == 4, "linewidth(1,4) sets dataset 1")
check(uc.sets[2].alpha == 0.4, "alpha(2,0.4) sets dataset 2")
check(uc.variable_formats == {}, "dataset-path leaves variable_formats empty")

uc.plot('x', 'Temp', suptitle='A. Dataset-level: red / green / blue datasets').show()

UniChart Notebook Environment Initialized.
Loaded Set 0: A
Loaded Set 1: B
Loaded Set 2: C
PASS: color(0,'red') sets dataset 0
PASS: color(1,'green') sets dataset 1
PASS: color(2,'blue') sets dataset 2
PASS: marker('all','o') sets every dataset
PASS: markersize(0,14) sets dataset 0
PASS: linewidth(1,4) sets dataset 1
PASS: alpha(2,0.4) sets dataset 2
PASS: dataset-path leaves variable_formats empty


## B. Negative indexing & list selectors

`-1` is the last dataset; `-len(sets)` is the first; out-of-range returns no
datasets; lists may mix positive and negative indices.

In [12]:
uc = make_nb()

check([d.title for d in uc._get_uset_slice(-1)] == ['C'], "-1 -> last dataset (C)")
check([d.title for d in uc._get_uset_slice(-3)] == ['A'], "-3 -> first dataset (A)")
check(uc._get_uset_slice(-4) == [], "-4 (out of range) -> []")
check([d.title for d in uc._get_uset_slice([0, -1])] == ['A', 'C'], "[0,-1] -> first + last")

uc.color(-1, 'magenta')
check(uc.sets[-1].color == 'magenta', "color(-1,...) styles the last dataset")
check(uc.sets[0].color != 'magenta', "color(-1,...) leaves the first dataset alone")

uc.markersize([0, -1], 20)
check(uc.sets[0].markersize == 20 and uc.sets[2].markersize == 20, "markersize([0,-1],20) hits first+last")
check(uc.sets[1].markersize == DEFAULTS['markersize'], "middle dataset untouched by [0,-1]")

UniChart Notebook Environment Initialized.
Loaded Set 0: A
Loaded Set 1: B
Loaded Set 2: C
PASS: -1 -> last dataset (C)
PASS: -3 -> first dataset (A)
PASS: -4 (out of range) -> []
PASS: [0,-1] -> first + last
PASS: color(-1,...) styles the last dataset
PASS: color(-1,...) leaves the first dataset alone
PASS: markersize([0,-1],20) hits first+last
PASS: middle dataset untouched by [0,-1]


In [16]:
uc.plot("Flow", "Press", by='sets')

## C. Variable-level formatting via autodetection (string target)

A **string** selector is autodetected as a variable name and routed to
`var_format`. It writes `variable_formats` and must **not** change any dataset
attribute. Variable styling is rendered with `plot_ymult`.

In [4]:
uv = make_nb()
colors_before = [d.color for d in uv.sets]   # per-dataset default palette colors

uv.color('Temp', 'crimson')
uv.color('Press', 'navy')
uv.marker('Temp', '^')
uv.linestyle('Press', ':')
uv.markersize('Temp', 16)
uv.linewidth('Press', 5)
uv.alpha('Temp', 0.5)

check(uv.variable_formats['Temp'] == {'color': 'crimson', 'marker': '^', 'markersize': 16, 'alpha': 0.5},
      "Temp variable override accumulates color/marker/markersize/alpha")
check(uv.variable_formats['Press'] == {'color': 'navy', 'linestyle': ':', 'linewidth': 5},
      "Press variable override accumulates color/linestyle/linewidth")
check([d.color for d in uv.sets] == colors_before,
      "variable-path leaves every dataset color unchanged")
check(all(d.markersize == DEFAULTS['markersize'] for d in uv.sets),
      "variable-path leaves every dataset markersize at default")

uv.plot_ymult('x', ['Temp', 'Press'], suptitle='C. Variable-level: Temp crimson ^, Press navy :').show()

UniChart Notebook Environment Initialized.
Loaded Set 0: A
Loaded Set 1: B
Loaded Set 2: C
PASS: Temp variable override accumulates color/marker/markersize/alpha
PASS: Press variable override accumulates color/linestyle/linewidth
PASS: variable-path leaves every dataset color unchanged
PASS: variable-path leaves every dataset markersize at default


## D. Precedence — variable override beats dataset attribute

With every dataset set to gray, an overridden variable still renders in its own
color, while an un-overridden variable falls back to the dataset gray.

In [5]:
uv.color('all', 'gray')   # dataset-level baseline

eff_temp = _resolve_var_format(uv.sets[0], 'Temp', uv.variable_formats)
eff_flow = _resolve_var_format(uv.sets[0], 'Flow', uv.variable_formats)  # no override
check(eff_temp['color'] == 'crimson', "Temp override wins over dataset gray")
check(eff_flow['color'] == 'gray', "un-overridden Flow falls back to dataset gray")

uv.plot_ymult('x', ['Temp', 'Press', 'Flow'],
              suptitle='D. Precedence: Temp/Press overridden, Flow = dataset gray').show()

PASS: Temp override wins over dataset gray
PASS: un-overridden Flow falls back to dataset gray


## E. List of variable names

In [6]:
uv.linewidth(['Temp', 'Press', 'Flow'], 7)
check(uv.variable_formats['Temp']['linewidth'] == 7
      and uv.variable_formats['Press']['linewidth'] == 7
      and uv.variable_formats['Flow']['linewidth'] == 7,
      "linewidth(['Temp','Press','Flow'],7) applies to all three variables")

PASS: linewidth(['Temp','Press','Flow'],7) applies to all three variables


## F. Int-normalization on both paths

An integer color resolves to the default Plotly palette; an integer marker
resolves through `marker_map`. This must behave identically whether the target
is a dataset or a variable. `'reset'` (a string) passes through untouched.

In [7]:
palette2 = px.colors.qualitative.Plotly[2]

# variable path
uv.color('Temp', 2)
uv.marker('Temp', 1)
check(uv.variable_formats['Temp']['color'] == palette2, "int color -> palette[2] (variable path)")
check(uv.variable_formats['Temp']['marker'] == marker_map(1), "int marker -> marker_map(1) (variable path)")

# dataset path
uc2 = make_nb()
uc2.color(0, 2)
uc2.marker(0, 1)
check(uc2.sets[0].color == palette2, "int color -> palette[2] (dataset path)")
check(uc2.sets[0].marker == marker_map(1), "int marker -> marker_map(1) (dataset path)")

PASS: int color -> palette[2] (variable path)
PASS: int marker -> marker_map(1) (variable path)
UniChart Notebook Environment Initialized.
Loaded Set 0: A
Loaded Set 1: B
Loaded Set 2: C
PASS: int color -> palette[2] (dataset path)
PASS: int marker -> marker_map(1) (dataset path)


## G. Reset paths

In [8]:
# 'reset' clears a single attribute via the setter
uv.color('Temp', 'reset')
check('color' not in uv.variable_formats['Temp'], "color('Temp','reset') clears only the color override")
check('marker' in uv.variable_formats['Temp'], "...other Temp overrides remain after color reset")

# clear an entire variable
uv.clear_var_format('Press')
check('Press' not in uv.variable_formats, "clear_var_format('Press') removes the whole entry")

# global reset
uv.reset_format()
check(uv.variable_formats == {}, "reset_format() clears all variable formatting")

PASS: color('Temp','reset') clears only the color override
PASS: ...other Temp overrides remain after color reset
PASS: clear_var_format('Press') removes the whole entry
Reset: dataset formatting, variable formats, lines, highlights, axis limits, font sizes.
PASS: reset_format() clears all variable formatting


## H. Dataset titles are no longer selectors

After the selector change, a title string is **not** resolved to a dataset — it
is treated as a variable name by the formatting setters.

In [9]:
uc3 = make_nb()
before = uc3.sets[0].color

check(uc3._get_uset_slice('A') == [], "_get_uset_slice('A') no longer matches the title")

uc3.color('A', 'orange')   # 'A' is a title, but is treated as a VARIABLE now
check('A' in uc3.variable_formats, "color('A',...) creates a variable override, not a dataset edit")
check(uc3.sets[0].color == before, "dataset 0 color unchanged by the title-string call")

UniChart Notebook Environment Initialized.
Loaded Set 0: A
Loaded Set 1: B
Loaded Set 2: C
PASS: _get_uset_slice('A') no longer matches the title
PASS: color('A',...) creates a variable override, not a dataset edit
PASS: dataset 0 color unchanged by the title-string call


## Summary

In [10]:
print(f'{PASSED} assertions passed.')
print('ALL FORMATTING TESTS PASSED' if PASSED >= 26 else 'CHECK COUNT UNEXPECTED')

34 assertions passed.
ALL FORMATTING TESTS PASSED
